In [4]:
# Tracker 02.06 — Target Data Generator

import pandas as pd
import numpy as np
from pathlib import Path

# ============================================================
# 1. INITIAL SETUP
# ============================================================

np.random.seed(42)

project_path = Path.cwd().parent
data_path = project_path / "data" / "synthetic"

stores_file = data_path / "stores_master.csv"
products_file = data_path / "products_master.csv"

print("Project path:", project_path)
print("Stores file:", stores_file)
print("Products file:", products_file)

if not stores_file.exists():
    raise FileNotFoundError("stores_master.csv was not found.")

if not products_file.exists():
    raise FileNotFoundError("products_master.csv was not found.")

# ============================================================
# 2. LOAD MASTER DATA
# ============================================================

stores_df = pd.read_csv(stores_file)
products_df = pd.read_csv(products_file)

print("\nMaster data loaded successfully.")
print("Stores shape:", stores_df.shape)
print("Products shape:", products_df.shape)

# ============================================================
# 3. VALIDATE REQUIRED COLUMNS
# ============================================================

if "store_id" not in stores_df.columns:
    raise ValueError("store_id column is missing from stores_master.csv")

if "category" not in products_df.columns:
    raise ValueError("category column is missing from products_master.csv")

print("\nRequired columns validated successfully.")

# ============================================================
# 4. GET UNIQUE STORES AND CATEGORIES
# ============================================================

stores_df = (
    stores_df
    .drop_duplicates(subset=["store_id"])
    .sort_values("store_id")
    .reset_index(drop=True)
)

available_store_ids = stores_df["store_id"].tolist()

categories = sorted(
    products_df["category"]
    .dropna()
    .unique()
    .tolist()
)

print("\nUnique stores:", len(available_store_ids))
print("Categories:", categories)
print("Total categories:", len(categories))

if len(available_store_ids) < 60:
    raise ValueError(
        f"Expected at least 60 stores, but found {len(available_store_ids)}."
    )

# ============================================================
# 5. CATEGORY DEMAND ASSUMPTIONS
# ============================================================

category_demand_weight = {
    "Electronics": 1.25,
    "Grocery": 1.20,
    "Fashion": 1.00,
    "Home & Kitchen": 0.90,
    "Beauty & Personal Care": 0.80,
    "Sports & Fitness": 0.85
}

# Apply default weight if a category is not listed
for category in categories:
    if category not in category_demand_weight:
        category_demand_weight[category] = 1.00

# ============================================================
# 6. CATEGORY MARGIN ASSUMPTIONS
# ============================================================

category_margin_range = {
    "Electronics": (0.18, 0.30),
    "Grocery": (0.08, 0.25),
    "Fashion": (0.35, 0.60),
    "Home & Kitchen": (0.18, 0.35),
    "Beauty & Personal Care": (0.30, 0.55),
    "Sports & Fitness": (0.25, 0.50)
}

for category in categories:
    if category not in category_margin_range:
        category_margin_range[category] = (0.15, 0.30)

# ============================================================
# 7. SEASONALITY
# ============================================================

seasonal_multiplier = {
    1: 0.95,
    2: 0.90,
    3: 1.00,
    4: 1.02,
    5: 1.05,
    6: 1.08,
    7: 1.03,
    8: 1.08,
    9: 1.15,
    10: 1.35,
    11: 1.45,
    12: 1.60
}

# ============================================================
# 8. STORE POTENTIAL
# ============================================================

store_potential = {}

for store_id in available_store_ids:
    store_potential[store_id] = np.random.lognormal(
        mean=0,
        sigma=0.35
    )

print("\nStore potential generated successfully.")

# ============================================================
# 9. ACTIVE STORE HISTORY
# ============================================================

def get_active_store_count(target_month):

    if target_month < pd.Timestamp("2024-07-01"):
        return 52

    elif target_month < pd.Timestamp("2025-01-01"):
        return 54

    elif target_month < pd.Timestamp("2025-07-01"):
        return 56

    elif target_month < pd.Timestamp("2026-01-01"):
        return 58

    elif target_month < pd.Timestamp("2026-05-01"):
        return 59

    else:
        return 60

# ============================================================
# 10. TARGET MONTHS
# ============================================================

target_months = pd.date_range(
    start="2024-01-01",
    end="2026-08-01",
    freq="MS"
)

print("\nTarget months:", len(target_months))
print("Start month:", target_months.min().date())
print("End month:", target_months.max().date())

# ============================================================
# 11. GENERATE TARGET DATA
# ============================================================

target_records = []

for target_month in target_months:

    active_store_count = get_active_store_count(target_month)

    active_store_ids = available_store_ids[:active_store_count]

    month_factor = seasonal_multiplier[target_month.month]

    # Yearly growth
    if target_month.year == 2024:
        year_growth = 1.00

    elif target_month.year == 2025:
        year_growth = 1.08

    else:
        year_growth = 1.16

    for store_id in active_store_ids:

        store_factor = store_potential[store_id]

        for category in categories:

            demand_factor = category_demand_weight[category]

            margin_low, margin_high = (
                category_margin_range[category]
            )

            expected_margin = np.random.uniform(
                margin_low,
                margin_high
            )

            # Base revenue target
            base_revenue = (
                2_500_000
                * store_factor
                * demand_factor
                * month_factor
                * year_growth
            )

            # Controlled variation
            target_noise = np.random.uniform(
                0.90,
                1.10
            )

            revenue_target = (
                base_revenue
                * target_noise
            )

            revenue_target = round(
                revenue_target,
                2
            )

            # Profit target
            profit_target = round(
                revenue_target
                * expected_margin,
                2
            )

            # Order target
            average_order_value = np.random.uniform(
                1800,
                2800
            )

            order_target = int(
                round(
                    revenue_target
                    / average_order_value
                )
            )

            # Return-rate target
            if category == "Fashion":

                return_rate_target = np.random.uniform(
                    8.0,
                    10.0
                )

            elif category == "Electronics":

                return_rate_target = np.random.uniform(
                    5.0,
                    7.0
                )

            elif category == "Home & Kitchen":

                return_rate_target = np.random.uniform(
                    4.0,
                    5.0
                )

            elif category == "Beauty & Personal Care":

                return_rate_target = np.random.uniform(
                    2.0,
                    4.0
                )

            elif category == "Grocery":

                return_rate_target = np.random.uniform(
                    0.5,
                    1.5
                )

            else:

                return_rate_target = np.random.uniform(
                    3.0,
                    7.0
                )

            return_rate_target = round(
                return_rate_target,
                2
            )

            # Category code for target ID
            category_code = "".join(
                word[0]
                for word in category.split()
                if word
            ).upper()[:4]

            # Target ID
            store_number = (
                str(store_id)
                .replace("STORE", "")
            )

            target_id = (
                f"TGT"
                f"{target_month.strftime('%Y%m')}"
                f"{store_number}"
                f"{category_code}"
            )

            target_records.append({
                "target_id": target_id,
                "target_month": target_month.date(),
                "store_id": store_id,
                "category": category,
                "revenue_target": revenue_target,
                "profit_target": profit_target,
                "order_target": order_target,
                "return_rate_target_pct": return_rate_target
            })

# ============================================================
# 12. CREATE DATAFRAME
# ============================================================

targets_df = pd.DataFrame(target_records)

print("\nTarget data generated successfully.")
print("Shape:", targets_df.shape)
print("Rows:", len(targets_df))
print("Columns:", len(targets_df.columns))

print("\nColumns:")
print(list(targets_df.columns))

# ============================================================
# 13. SAVE TARGET DATASET
# ============================================================

output_path = data_path / "targets.csv"

targets_df.to_csv(
    output_path,
    index=False
)

print("\nTarget dataset saved successfully.")
print("File:", output_path)
print("Rows saved:", len(targets_df))

# ============================================================
# 14. RELOAD SAVED DATA
# ============================================================

saved_targets_df = pd.read_csv(
    output_path
)

print("\nSaved file loaded successfully.")
print("Saved shape:", saved_targets_df.shape)

# ============================================================
# 15. FINAL VALIDATION
# ============================================================

print("\n==========================================")
print("FINAL TARGET DATA VERIFICATION")
print("==========================================")

print(
    "Rows:",
    len(saved_targets_df)
)

print(
    "Columns:",
    len(saved_targets_df.columns)
)

print(
    "Unique Target IDs:",
    saved_targets_df["target_id"].nunique()
)

print(
    "Duplicate Target IDs:",
    saved_targets_df["target_id"].duplicated().sum()
)

print(
    "Missing Values:",
    saved_targets_df.isnull().sum().sum()
)

print(
    "Unique Stores:",
    saved_targets_df["store_id"].nunique()
)

print(
    "Unique Categories:",
    saved_targets_df["category"].nunique()
)

print(
    "Unique Months:",
    saved_targets_df["target_month"].nunique()
)

# ============================================================
# 16. VALIDATE TARGET GRAIN
# ============================================================

grain_duplicates = (
    saved_targets_df
    .duplicated(
        subset=[
            "target_month",
            "store_id",
            "category"
        ]
    )
    .sum()
)

print(
    "Duplicate Month-Store-Category combinations:",
    grain_duplicates
)

# ============================================================
# 17. VALIDATE NUMERIC VALUES
# ============================================================

print(
    "Minimum Revenue Target:",
    saved_targets_df["revenue_target"].min()
)

print(
    "Minimum Profit Target:",
    saved_targets_df["profit_target"].min()
)

print(
    "Minimum Order Target:",
    saved_targets_df["order_target"].min()
)

print(
    "Minimum Return Rate Target:",
    saved_targets_df["return_rate_target_pct"].min()
)

# ============================================================
# 18. DISPLAY SAMPLE
# ============================================================

print("\nSample Target Records:")

display(
    saved_targets_df.head(10)
)

# ============================================================
# 19. COMPLETION MESSAGE
# ============================================================

if (
    saved_targets_df["target_id"].duplicated().sum() == 0
    and saved_targets_df.isnull().sum().sum() == 0
    and grain_duplicates == 0
):

    print("\n==========================================")
    print("02.06 TARGET DATA GENERATION COMPLETED")
    print("==========================================")

else:

    print("\nValidation requires attention.")

Project path: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics
Stores file: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\stores_master.csv
Products file: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\products_master.csv

Master data loaded successfully.
Stores shape: (60, 11)
Products shape: (2500, 16)

Required columns validated successfully.

Unique stores: 60
Categories: ['Beauty & Personal Care', 'Electronics', 'Fashion', 'Grocery', 'Home & Kitchen']
Total categories: 5

Store potential generated successfully.

Target months: 32
Start month: 2024-01-01
End month: 2026-08-01

Target data generated successfully.
Shape: (8980, 8)
Rows: 8980
Columns: 8

Columns:
['target_id', 'target_month', 'store_id', 'category', 'revenue_target', 'profit_target', 'order_target', 'return_rate_target_pct']

Target dataset saved successfully.
File: c:\Users\HP\OneDrive\Desktop\retail_sales_analytics\data\synthetic\targets.csv
Rows saved: 8980

Saved file lo

,target_id,target_month,store_id,category,revenue_target,profit_target,order_target,return_rate_target_pct
0,TGT20240100001B&PC,2024-01-01,STORE00001,Beauty & Personal Care,2063427.81,789724.39,977,2.65
1,TGT20240100001E,2024-01-01,STORE00001,Electronics,3629630.30,971117.54,1351,5.94
2,TGT20240100001F,2024-01-01,STORE00001,Fashion,2946482.07,1119364.30,1151,9.12
3,TGT20240100001G,2024-01-01,STORE00001,Grocery,3386941.48,714862.84,1458,0.93
4,TGT20240100001H&K,2024-01-01,STORE00001,Home & Kitchen,2343907.30,432031.93,1280,4.64
5,TGT20240100002B&PC,2024-01-01,STORE00002,Beauty & Personal Care,1813346.53,686513.04,670,2.50
6,TGT20240100002E,2024-01-01,STORE00002,Electronics,2973071.07,681564.50,1465,5.15
7,TGT20240100002F,2024-01-01,STORE00002,Fashion,2109486.41,891126.93,773,9.62
8,TGT20240100002G,2024-01-01,STORE00002,Grocery,2917095.53,547476.52,1120,0.69
9,TGT20240100002H&K,2024-01-01,STORE00002,Home & Kitchen,2052548.25,680902.15,787,4.90



02.06 TARGET DATA GENERATION COMPLETED
